# Async Python (asyncio)

### Short Version

* all function should start with async 
* all calls should start with await 

### Long Version

To be able to do things works in paralell.

* Asyncio provides lightweight alternative to threading or multiprocessing
* Functions defines with aync def are called __coroutine__ - they are special functions that can be paused and resumed
* Calling a coroutine doens't execute it immediately - it returns a coroutine object
* To actually run a coroutine, you must await it, which schedules it for a execution within an event loop.
* While a coroutine is waiting, the event loop can run other coroutines.

using await asyncio.gather(<br>
&nbsp;&nbsp;do_some_processing(),<br>
&nbsp;&nbsp;do_other_processing(),<br>
&nbsp;&nbsp;do_yet_more_processing()<br>
)

is the way to run thing somewhat parallely

# Introducing OpenAI Agents SDK

### Initial terminology
* __Agents__ represent a particular LLM with system prompt and tools

#### How does agent collaborate- 
* __Agents as tools__ nad __handoffs__ for collaboration

#### What if agent is untameable
* Use guardrails

### Three steps

1. Create an instance of __Agent__
2. User with trace() to track the agent. (Can see all activities in log)
3. Call Runner.run() to run one 'Application Level Turn'. The runner keeps loopiong until it reaches stopping point. ()

#### With Coding Agent: ALWAYS BE THE BOSS, PERSON IN DRIVING SEAT
1. invest in your prompts: Be precissem ensure current APIs, Demand conciseness- no slop!!
2. start with simple task, small piece work on
3. work incrementally, test constantly, validate sucess criteria
4. don't get lazy -challenge & demand evidence
5. handle frustration with style.

### Let's Start

#### Part 1: simple Agent and Agent loop

In [43]:
from dotenv import load_dotenv
import os, requests

# this is new- 
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession

In [ ]:
load_dotenv(override = True)

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = os.getenv("PUSHOVER_URL")

In [30]:
#  Make an instance of agent
agent = Agent(name = "Jokester", instructions = "you are a dad joke teller", model = 'gpt-5.4-mini')

In [31]:
# Run the joke with Runner.run(agent, prompt)
results = await Runner.run(agent, "Tell us a joke about ldr, and then explain joke")

In [32]:
print(results.final_output)

Why did the LDR bring a ladder to the relationship?

Because it wanted to *bridge the gap*.

Explanation: “LDR” means long-distance relationship, so the joke is that the relationship is far apart. A ladder doesn’t really fix distance, but “bridge the gap” is a phrase meaning to reduce separation—so it’s a silly pun.


In [33]:
results.to_input_list()

[{'content': 'Tell us a joke about ldr, and then explain joke',
  'role': 'user'},
 {'id': 'msg_0ae4f71f8239d57d006b048e1826b887d29b083716fb9176ca',
  'content': [{'annotations': [],
    'text': 'Why did the LDR bring a ladder to the relationship?\n\nBecause it wanted to *bridge the gap*.\n\nExplanation: “LDR” means long-distance relationship, so the joke is that the relationship is far apart. A ladder doesn’t really fix distance, but “bridge the gap” is a phrase meaning to reduce separation—so it’s a silly pun.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

Tracing/ observability is part of SDK that traces the agent framework not the LLM

In [34]:
# Adding observability with a trace
with trace("Tell a joke"):
    result = await Runner.run(agent, "tell a sad fact")
print(result.final_output)

A sad fact: honey never spoils—but some of us do.


to check we need to go to - https://platform.openai.com/traces

Streaming- is when you call LLM and rather than waiting and getting back the whole result,you let it gradually flow the results back to you and uses a technology called SSE 

In [35]:
results = Runner.run_streamed(agent, "Please tell me 5 jokes about AI agents")
async for event in results.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end = "", flush = True)

1. Why did the AI agent bring a ladder?  
   Because it wanted to reach the next level of intelligence.

2. Why do AI agents make terrible comedians?  
   They always need a prompt.

3. What do you call an AI agent that loves gardening?  
   A plot bot.

4. Why was the AI agent always calm?  
   It had excellent state management.

5. Why did the AI agent break up with the chatbot?  
   It said, “You’re just not my type of token.”

#### Part 2: Adding a tool

In [45]:
if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [46]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data = payload)

In [47]:
push("Hey!")

Push: Hey!


In [48]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [39]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x00000217B45B2E10>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [40]:
push_tool.params_json_schema

{'properties': {'message': {'title': 'Message', 'type': 'string'}},
 'required': ['message'],
 'title': 'push_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [49]:
notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])

In [ ]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)

Notified


#### Part 3: Session (Memory)

* Within Runner.run() application level turn, the conversation history is maintained.
* But each call to Runner.run() is a fresh start

In [55]:
agent = Agent(name = "Assistant", model = "gpt-5.4-mini")

In [56]:
response = await Runner.run(agent, "Hi there, my name is Akhil")
print(response.final_output)

Hi Akhil! Nice to meet you. How can I help?


In [57]:
response = await Runner.run(agent, "what is my name")
print(response.final_output)

I don’t know your name yet. If you want, tell me and I’ll use it.


##### Memory Approach 1 - Just Manually pass in list of dictionaries

In [58]:
response = await Runner.run(agent, "Hi there, my name is Akhil")
print(response.final_output)

Hi Akhil! Nice to meet you. How can I help today?


In [59]:
response.to_input_list()

[{'content': 'Hi there, my name is Akhil', 'role': 'user'},
 {'id': 'msg_03bfc9beabf2cba5006b0494423d2087d281152dc4905d94b2',
  'content': [{'annotations': [],
    'text': 'Hi Akhil! Nice to meet you. How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [60]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name"}]
next_input

[{'content': 'Hi there, my name is Akhil', 'role': 'user'},
 {'id': 'msg_03bfc9beabf2cba5006b0494423d2087d281152dc4905d94b2',
  'content': [{'annotations': [],
    'text': 'Hi Akhil! Nice to meet you. How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name"}]

In [61]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Akhil.


##### Memory Approach 2 - Use OpenAI Agents SDK built in SQLite Session

In [62]:
# This is created in memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")
session = SQLiteSession("12345")

In [63]:
response = await Runner.run(agent, "Hi, My name is Akhil", session = session)
print(response.final_output)

Hi Akhil — nice to meet you! How can I help?


In [64]:
response = await Runner.run(agent, "what is my name", session = session)
print(response.final_output)

Your name is Akhil.
